In [ ]:
import requests
from bs4 import BeautifulSoup
import os
from google import genai
from google.genai import types
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
# response = requests.get('https://www.merit-times.com.tw/NewsPage.aspx?unid=903451')
response = requests.get('https://rate.bot.com.tw/xrt?Lang=zh-TW')
soup = BeautifulSoup(response.text,"html.parser")
body_content = soup.body
for script_or_style in body_content(['script','style']):
    script_or_style.extract()

table_content = body_content.find(title='牌告匯率')
table_lines = table_content.get_text(separator='\n')
cleaned_content = "\n".join(
    [line.strip() for line in table_lines.splitlines() if line.strip()]
    )


system_instruction = '''
    你的任務是取出指定的內容,並輸出文字
    請依照下面的指示:
    樣本:
    ```
    澳幣 (AUD)
    澳幣 (AUD)
    20.15
    20.93
    20.365
    20.71
    ```
    輸出的格式:
    澳幣(AUD):
    現金匯率(本行買入):20.15
    現金匯率(本行賣出):20.93
    即期匯率(本行買入):20.365
    即期匯率(本行賣出):20.365 
    ==================   
    '''

response = client.models.generate_content(
    model="gemini-flash-latest",
    contents=cleaned_content,
    config=types.GenerateContentConfig(system_instruction=system_instruction)
)
result_text = response.text
print(result_text)

In [ ]:
from google import genai
from google.genai import types
from pydantic import BaseModel

class Rate(BaseModel):
    buy:float | None
    sell:float | None

class Recipe(BaseModel):
    幣別:str
    類別1:Rate
    類別2:Rate

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

system_instruction = '''    
    1.如果無法轉成float,請使用None替代
    **輸入的格式:**
    ```
    澳幣(AUD):
    現金匯率(本行買入):20.15
    現金匯率(本行賣出):20.93
    即期匯率(本行買入):20.365
    即期匯率(本行賣出):20.71 
    ```

    **轉換輸入的格式成為輸出json schema:**
    ```
    [
        {
           '幣別':'澳幣',
           '類別1':{
                    'buy':20.15,
                    'sell':20.93
                }
           '類別2':{
                    'buy':20.365,
                    'sell':20.71
                } 
        }
    ]
    ```
    '''

response = client.models.generate_content(
    model="gemini-flash-latest",
    contents=result_text,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        response_mime_type="application/json",
        response_schema=list[Recipe]
    )
)
print(response.text)